# 1. Basic implementations <br>
    a. load doc 
    b. chunk it
    c. embed

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
import os
from dotenv import load_dotenv

#load key for hugging face
load_dotenv()
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

#1. load the doc
loader = PyPDFLoader("attention.pdf")
pdfDoc = loader.load()
print(pdfDoc)

#2. Chunking docs
seperators = ["\n\n","\n"," ",""]
pdfSplitter = RecursiveCharacterTextSplitter(separators=seperators,chunk_size = 500, chunk_overlap = 50)
pdfChunks = pdfSplitter.split_documents(pdfDoc)
print(len(pdfChunks))
pdfChunksText = [doc.page_content for doc in pdfChunks]

#3. Embeddings
pdfEmbeder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
pdfEmbeddings = pdfEmbeder.embed_documents(pdfChunksText)
print(len(pdfEmbeddings))



# 2. FAISS basics 
1. from_documents()
2. from_embeddings()
3. similarity_search
4. similarity_search_by_vector
5. similarity_search_with_score
6. similarity_search_with_score_by_vector
5. similarity_search_with_relevance_scores

Document objects, not raw strings — loaders and splitters work with LangChain's Document class (page_content + metadata), not plain str. You <br>typically don't call embed_documents directly yourself in a real pipeline — that's usually handled internally by the vector store's .<br>from_documents() method. But understanding it explicitly, like you're doing, is the right way to learn it before the abstraction hides it.<br>

In [ ]:
from langchain_community.vectorstores import FAISS

# 1. from_documents
vectorStore = FAISS.from_documents(pdfChunks,embedding=pdfEmbeder)
answersSS = vectorStore.similarity_search(query="What is attention mechanism?",k=1)
print("similarity_search:",answersSS[0].page_content)
answersSSV = FAISS.similarity_search_by_vector(self=vectorStore,embedding=pdfEmbeder.embed_query("What is attention mechanism?"),k=1)
print("similarity_search_by_vector:",answersSSV[0].page_content)
answersSSS = FAISS.similarity_search_with_score(self=vectorStore,query="What is attention mechanism?",k=3)
print("similarity_search_with_score",answersSSS)
answersSSSV = FAISS.similarity_search_with_score_by_vector(self=vectorStore,embedding=pdfEmbeder.embed_query("What is attention mechanism?"),k=1)
print("similarity_search_with_score_by_vector",answersSSSV[0])
answersSSRS= FAISS.similarity_search_with_relevance_scores(self=vectorStore,query="What is attention mechanism?",k=1)
print("similarity_search_with_relevance_scores",answersSSRS[0][1])








In [ ]:
#2. from_embeddings  -> uses only page content, no metadata. Metadata explaicitly added.
text_embedDocs = list(zip(pdfChunksText,pdfEmbeddings))
vectorStore = FAISS.from_embeddings(text_embeddings=text_embedDocs,embedding=pdfEmbeder,metadatas=[{"page" : i} for i in range(len(pdfChunksText))])
answers = vectorStore.similarity_search(query="What is attention mechanism?",k=1)
print(answers[0].metadata)

# MMR
1. max_marginal_relevance_search
2. max_marginal_relevance_search_by_vector

In [ ]:
answerMMR=FAISS.max_marginal_relevance_search(self=vectorStore,query="What is attention mechanism?",k=3)
answerMMRS = FAISS.max_marginal_relevance_search_by_vector(self=vectorStore, embedding=pdfEmbeder.embed_query("What is attention mechanism?"),k=4)
print(answerMMR[0].metadata)
print(answerMMRS[0].metadata)

# metdata filtering

1. No score version for MMR. Unlike similarity_search_with_score, there's no max_marginal_relevance_search_with_score. MMR is a re-ranking algorithm on top of similarity — it picks a diverse subset from a larger candidate pool (fetch_k, default 20), so the "score" concept doesn't map cleanly the same way. If you need scores alongside diverse results, you'd have to fetch the candidates yourself and reason about ranks, not raw scores.<br>
2. k vs fetch_k —  it defaults to 20. That means it pulls the 20 nearest neighbors internally, then greedily selects 4 of those that are relevant and mutually diverse. Worth explicitly printing what fetch_k=20 vs fetch_k=4 (i.e., barely any pool to select from) does to your output — that'll make the redundancy-reduction effect visible.

In [1]:
filter_data = {"producer":"pdfTeX-1.40.25"}

answerFiltData = vectorStore.similarity_search(query="What is attention mechanism?",k=3,fetch_k=10,filter=filter_data)
print(len(answerFiltData))

NameError: name 'vectorStore' is not defined

# Persistence
save_local()/ load_local()

In [ ]:
vectorStore.save_local(folder_path="/Users/sanvijanvi/AI_Workspace/ML_DL_Projects/genAI_langchain_projects/langchain/langchain_basics")

In [ ]:
vectorStore = FAISS.load_local(folder_path="/Users/sanvijanvi/AI_Workspace/ML_DL_Projects/genAI_langchain_projects/langchain/langchain_basics",embeddings=pdfEmbeder,
                             allow_dangerous_deserialization=True  )

# Incremental indexing — add_documents() / add_texts() / merge_from()

In [ ]:
type(vectorStore.index)

In [ ]:
vectorStore.index.ntotal

In [ ]:
from langchain_core.documents import Document
documents = [
    "Machine learning is a subset of artificial intelligence.",
    "Deep learning uses neural networks.",
    "Python is used for data science.",
    "Cooking requires different ingredients.",
    "Football is a popular sport."
]

listDocs = [Document(page_content= text) for text in documents]

vectorStore.add_documents(documents=listDocs)
vectorStore.index.ntotal

#add_text similar to above, it needs list of text
#add_embedding needs list of tupe(text, emebedding)
# vectorStore.merge_from(another vector store)

# Indexing
1. index_to_docstore_id
2. vectorStore.index.ntotal
3. vectorStore.docstore.search
4. reconstruct()

In [ ]:
print(f"length of indixes:{vectorStore.index.ntotal}")

indicesList = list(range(vectorStore.index.ntotal))
print(indicesList)

# Mapping FAISS's internal integer IDs to LangChain's docstore IDs
print(vectorStore.index_to_docstore_id)

# Getting the actual Document (content + metadata) for each vector

In [ ]:
for faiss_id, doc_id in vectorStore.index_to_docstore_id.items():
    doc = vectorStore.docstore.search(doc_id)
    print(faiss_id, "->", doc_id, "->", doc.page_content[:50], doc.metadata)

# Reconstructing the actual raw vector
Note: reconstruct() only works on index types that support it (like IndexFlatL2 — which is what you have) — some approximate indexes (like certain IVF variants without make_direct_map()) don't support reconstruction without extra setup.

In [ ]:
print(vectorStore.index.reconstruct(0)) # gets the embedding for FAISS internal id 0

# Index types FAISS
1. IndexFlatL2  - 	Exact search, Euclidean distance  - Default, small datasets, ground-truth benchmarking
2. IndexFlatIP - 	Exact search, inner product  - Use when vectors are normalized (cosine similarity becomes inner product)
3. IndexIVFFlat - 	Approximate — clusters vectors into cells, searches only nearby cells - cells, searches only nearby cells	Larger datasets (100k+), needs .train() before adding vectors
4. IndexHNSWFlat - Approximate — graph-based nearest neighbor - Fast queries, good recall, higher memory

In [ ]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore

embeddingDim = len(pdfEmbeder.embed_query("What is this?"))
index = faiss.IndexFlatL2(embeddingDim)
vectorStoreFlat = FAISS(embedding_function=pdfEmbeder,docstore=InMemoryDocstore(),index_to_docstore_id={},index=index)

docs = vectorStoreFlat.add_documents(pdfChunks)
print(docs[0])

# Notes:
1. fetch_k only matters when a filter is passed
2. Order is ascending by distance, not "descending." FAISS's IndexFlatL2 returns nearest neighbors ranked by L2 distance — lowest first

#    Chroma DB

You can create multiple Chroma collections, but each collection is separate. You don't put two collections inside one Chroma vectorstore object.
But, we can create one chunk from 2 chunks and create one chroma collection.

The include parameter controls what additional data Chroma returns.
collection_metadata - This metadata describes the collection itself, not each PDF chunk.

In [ ]:
from langchain_community.vectorstores import Chroma

ids = [f"doc_{i}" for i in range(len(pdfChunks))]
vectorStoreChroma = Chroma.from_documents(documents=pdfChunks,collection_name="Example",ids=ids,persist_directory="./chroma_test1",embedding=pdfEmbeder)
collection = vectorStoreChroma._collection
collection.get()
print(collection.name, collection.get().keys(),collection.get(include = ["embeddings"])["embeddings"])

collection.query() performs vector similarity search under the hood.

In [ ]:
answersChroma = vectorStoreChroma.similarity_search(query="What is attention mechanism?",k=3)
print(answersChroma[0].metadata)
answersChroma = vectorStoreChroma.similarity_search_by_vector(pdfEmbeder.embed_query("What is attention mechanism?"),k=3)
print(answersChroma[0])
answersChroma = vectorStoreChroma.similarity_search_with_score(query="What is attention mechanism?",k=3)
print(answersChroma[0][1])
answersChroma = vectorStoreChroma.max_marginal_relevance_search(query="What is attention mechanism?",k=3,fetch_k=10,filter={"page":13})
print(answersChroma[0])


# Filtering :
Chroma has native metadata filtering/query capabilities, while LangChain's FAISS vector store implements metadata filtering around the FAISS index and associated document store. Therefore, the exact filtering behavior/API depends on the LangChain FAISS implementation.<br>
1. $eq   equal
2. $ne   not equal
3. $gt   greater than
4. $gte  greater than or equal
5. $lt   less than
6. $lte  less than or equal
7. $in   matches any value in a list
8. $nin  doesn't match values in a list
9. $and
10. $or
11. $contains

In [ ]:
answers = vectorStoreChroma.similarity_search(query="Exaplain multihead attention", k =3,filter={
    "page":{"$gte":14}
})

print([(doc.metadata)["page_label"] for doc in answers])

answers = vectorStoreChroma.max_marginal_relevance_search(query="Exaplain multihead attention", k =3,filter={
    "page":{"$gte":14}
})

print([(doc.metadata)["page_label"] for doc in answers])

# Note : 
FAISS is primarily a vector similarity-search library, whereas Chroma provides a more complete vector-store/database layer, including native metadata filtering, persistence, collections, etc.<br>

you generally don't need to call a separate persist() method with current Chroma versions. The update is persisted automatically.

# collection
Chroma <br>
├── from_documents()<br>
├── from_texts()<br>
├── collection<br>
├── collection.get() ---> include, where,where_docuemnts,ids<br> 
├── collection.query()  --> query_texts, n_results, include, where,where_document<br>
├── collection.add()<br>
├── collection.delete()<br>
├── collection.update()<br>
├── collection_name<br>
├── persist_directory<br>
└── reopening/loading existing collections<br>

In [ ]:
print(collection.get()["ids"])

answers = collection.query(query_texts=["What is attention?","Explain multihead attention"],n_results=2,include=["embeddings"])
print(answers["ids"])

documents = [
    "Machine learning is a subset of artificial intelligence.",
    "Deep learning uses neural networks.",
    "Python is used for data science.",
    "Cooking requires different ingredients.",
    "Football is a popular sport."
]
print(collection.get(include = ["documents"],ids=["doc_108"]))
print(collection.count())
id_documents = [f"doc_{i}" for i in range(collection.count(),collection.count()+len(documents))]
collection.add(ids=id_documents,documents=documents,embeddings=pdfEmbeder.embed_documents(documents))
print(collection.count())


In [ ]:
print(collection.get(include = ["documents"],ids=["doc_108"]))
collection.update(ids=["doc_108"],embeddings=pdfEmbeder.embed_documents(["Cricket is popular in India"]),documents=["Cricket is popular in India"])

print(collection.get(include=["documents"],ids=["doc_108"]))





In [ ]:
result = collection.delete(ids = "doc_107")
print(result)

In [ ]:
print(collection.get(include=["documents"],ids=["doc_107"]))

In [ ]:
del collection
del vectorStoreChroma

# load vector store chroma - persistence

Note : Collection name should be same as previous one

In [ ]:
vectorStoreChroma2 = Chroma(collection_name="Example",embedding_function=pdfEmbeder,persist_directory="./chroma_test1")
collection = vectorStoreChroma2._collection
print(collection.get(ids=["doc_108"],include=["documents"]))

In [ ]:
result = collection.get(
    where={"page": {"$lte": 5}}
)
print(result["metadatas"])

# where_document
where filters based on metadata, whereas where_document filters based on the actual document content.

In [ ]:
result = collection.get(
    where_document={"$contains": "attention"}
)

print(result["documents"])

In [ ]:
result = collection.query(
    query_texts=["What is attention?"],
    n_results=3,
    where={"page": {"$gte": 10}}
)

print(result["ids"])
print(result["distances"])
print(result["documents"])
print(result["metadatas"])

print(result.keys())
print(result["metadatas"][0][0]["page"])
print(result["metadatas"][0][1]["page"])
print(result["metadatas"][0][2]["page"])
